In [1]:
#CRIAÇÃO DA TABELA SILVER INVENTORY MOVEMENTS
from datetime import datetime
from pathlib import Path
import duckdb

# Procura o banco automaticamente : o notebook continuará encontrando o banco automaticamente dentro do projeto, 
# mesmo que ele seja movido para outra pasta ou máquina, desde que a estrutura do projeto seja mantida.
base_dir = Path.cwd().parent

db_path = next(base_dir.rglob("supply_chain_analytics.duckdb"))

conn = duckdb.connect(str(db_path))

# ============================================================
# CRIAÇÃO DA TABELA SILVER INVENTORY MOVEMENTS
# ============================================================
#
# Objetivo:
# Criar a tabela silver__inventory_movements__lite a partir da
# camada Bronze, aplicando regras de limpeza e padronização.
#
# Transformações realizadas:
# - Remoção de espaços extras (TRIM)
# - Padronização para letras maiúsculas (UPPER)
# - Conversão segura da coluna de data
# - Preservação das demais colunas necessárias para análise
#
# ============================================================

conn.execute("""
CREATE OR REPLACE TABLE silver__inventory_movements__lite AS

-- ============================================================
-- CTE responsável pela limpeza inicial dos dados
-- ============================================================
WITH base AS (

    SELECT

        -- Remove espaços extras e converte o ID para maiúsculo
        UPPER(TRIM(movement_id)) AS movement_id,

        -- Mantém a data original para tratamento posterior
        TRIM(movement_date) AS movement_date,

        -- Padroniza o tipo de movimentação
        UPPER(TRIM(movement_type)) AS movement_type,

        -- Padroniza o SKU do produto
        UPPER(TRIM(sku)) AS sku,

        -- Padroniza a localização de origem
        UPPER(TRIM(from_location_id)) AS from_location_id,

        -- Padroniza a localização de destino
        UPPER(TRIM(to_location_id)) AS to_location_id,

        -- Quantidade movimentada
        qty

    FROM bronze__inventory_movements__lite
)

-- ============================================================
-- Seleção final dos dados tratados
-- ============================================================
SELECT

    -- Identificador da movimentação
    movement_id,

    -- Conversão segura da data
    -- Suporta múltiplos formatos encontrados na camada Bronze:
    --
    -- 2026-01-10
    -- 2026-02-13 09:10
    -- 10/11/2026
    -- 09/01/2026 10:23
    --
    -- Caso nenhum formato seja compatível, retorna NULL
    COALESCE(

        TRY_CAST(movement_date AS TIMESTAMP),

        TRY_STRPTIME(movement_date, '%d/%m/%Y'),

        TRY_STRPTIME(movement_date, '%d/%m/%Y %H:%M'),

        TRY_STRPTIME(movement_date, '%Y-%m-%d'),

        TRY_STRPTIME(movement_date, '%Y-%m-%d %H:%M')

    ) AS movement_date,

    -- Tipo da movimentação
    movement_type,

    -- SKU do produto
    sku,

    -- Local de origem
    from_location_id,

    -- Local de destino
    to_location_id,

    -- Quantidade movimentada
    qty

FROM base;
""")


# Verifica se a tabela foi criada com sucesso
tables = conn.sql("SHOW TABLES").df()

if "silver__inventory_movements__lite" in tables["name"].values:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"✅ Tabela silver__inventory_movements__lite criada com sucesso!"
    )
else:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"❌ Tabela silver__inventory_movements__lite não foi criada!"
    )
conn.close()

[03/06/2026 23:39:10] ✅ Tabela silver__inventory_movements__lite criada com sucesso!


In [2]:
#CRIAÇÃO DA TABELA SILVER INVENTORY SNAPSHOTS
from datetime import datetime
from pathlib import Path
import duckdb

# Procura o banco automaticamente : o notebook continuará encontrando o banco automaticamente dentro do projeto, 
# mesmo que ele seja movido para outra pasta ou máquina, desde que a estrutura do projeto seja mantida.
base_dir = Path.cwd().parent

db_path = next(base_dir.rglob("supply_chain_analytics.duckdb"))

conn = duckdb.connect(str(db_path))

# ============================================================
# CRIAÇÃO DA TABELA SILVER INVENTORY SNAPSHOTS
# ============================================================
#
# Objetivo:
# Criar a tabela silver__inventory_snapshots__lite a partir da
# camada Bronze, aplicando regras de limpeza e padronização
# para os registros de estoque.
#
# Transformações realizadas:
# - Padronização dos identificadores de localização
# - Padronização dos códigos de produto (SKU)
# - Remoção de espaços excedentes
# - Conversão segura da data de snapshot
# - Preservação da quantidade disponível em estoque
#
# Esta tabela servirá como base para análises de posição de
# estoque, cobertura, rupturas e indicadores logísticos.
#
# ============================================================

conn.execute("""
CREATE OR REPLACE TABLE silver__inventory_snapshots__lite AS

-- ============================================================
-- CTE responsável pela limpeza inicial dos dados
-- ============================================================
WITH base AS (

    SELECT

        -- Data original do snapshot para tratamento posterior
        snapshot_date,

        -- Padroniza o identificador da localização
        -- removendo espaços extras e convertendo para maiúsculo
        UPPER(TRIM(location_id)) AS location_id,

        -- Padroniza o SKU do produto
        -- removendo espaços extras e convertendo para maiúsculo
        UPPER(TRIM(sku)) AS sku,

        -- Quantidade disponível em estoque no momento do snapshot
        on_hand_qty

    FROM bronze__inventory_snapshots__lite
)

-- ============================================================
-- Seleção final dos dados tratados
-- ============================================================
SELECT

    -- Conversão segura da data do snapshot
    -- Tenta converter diretamente para DATE e,
    -- caso não seja possível, tenta interpretar
    -- o formato DD-MM-YYYY
    COALESCE(
        TRY_CAST(snapshot_date AS TIMESTAMP),

        TRY_STRPTIME(snapshot_date, '%d/%m/%Y'),

        TRY_STRPTIME(snapshot_date, '%d/%m/%Y %H:%M'),

        TRY_STRPTIME(snapshot_date, '%Y-%m-%d'),

        TRY_STRPTIME(snapshot_date, '%Y-%m-%d %H:%M')
        
    ) AS snapshot_date,

    -- Identificador da localização do estoque
    location_id,

    -- Código do produto
    sku,

    -- Quantidade disponível em estoque
    on_hand_qty

FROM base;
""")

# Verifica se a tabela foi criada com sucesso
tables = conn.sql("SHOW TABLES").df()

if "silver__inventory_snapshots__lite" in tables["name"].values:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"✅ Tabela silver__inventory_snapshots__lite criada com sucesso!"
    )
else:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"❌ Tabela silver__inventory_snapshots__lite não foi criada!"
    )
conn.close()

[03/06/2026 23:39:10] ✅ Tabela silver__inventory_snapshots__lite criada com sucesso!


In [3]:
#CRIAÇÃO DA TABELA SILVER LOCATIOIN MASTER
from datetime import datetime
from pathlib import Path
import duckdb

# Procura o banco automaticamente : o notebook continuará encontrando o banco automaticamente dentro do projeto, 
# mesmo que ele seja movido para outra pasta ou máquina, desde que a estrutura do projeto seja mantida.
base_dir = Path.cwd().parent

db_path = next(base_dir.rglob("supply_chain_analytics.duckdb"))

conn = duckdb.connect(str(db_path))


# ============================================================
# CRIAÇÃO DA TABELA SILVER LOCATION MASTER
# ============================================================
#
# Objetivo:
# Criar a tabela silver__location_master__lite a partir da
# camada Bronze, aplicando regras de padronização nos dados
# cadastrais das localidades da operação logística.
#
# Transformações realizadas:
# - Remoção de espaços excedentes
# - Padronização de identificadores para letras maiúsculas
# - Padronização dos nomes das localidades
# - Padronização dos tipos de localização
# - Preservação da região geográfica
#
# Esta tabela funcionará como dimensão de localidades,
# permitindo análises por centros de distribuição,
# armazéns, lojas e regiões operacionais.
#
# ============================================================

conn.execute("""
CREATE OR REPLACE TABLE silver__location_master__lite AS

SELECT

    -- Identificador único da localização
    -- Padronizado para evitar inconsistências em joins
    UPPER(TRIM(location_id)) AS location_id,

    -- Nome da localização
    -- Padronizado para facilitar consultas e análises
    TRIM(location_name) AS location_name,

    -- Tipo da localização
    -- Exemplo: Warehouse, Store, Distribution Center
    UPPER(TRIM(location_type)) AS location_type,

    -- Região geográfica associada à localização
    -- Apenas remove espaços excedentes, preservando a grafia
    TRIM(region) AS region

FROM bronze__location_master__lite;
""")

# Verifica se a tabela foi criada com sucesso
tables = conn.sql("SHOW TABLES").df()

if "silver__location_master__lite" in tables["name"].values:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"✅ Tabela silver__location_master__lite criada com sucesso!"
    )
else:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"❌ Tabela silver__location_master__lite não foi criada!"
    )
conn.close()

[03/06/2026 23:39:10] ✅ Tabela silver__location_master__lite criada com sucesso!


In [4]:
#CRIAÇÃO DA TABELA SILVER PRODUCT MASTER
from datetime import datetime
from pathlib import Path
import duckdb

# Procura o banco automaticamente : o notebook continuará encontrando o banco automaticamente dentro do projeto, 
# mesmo que ele seja movido para outra pasta ou máquina, desde que a estrutura do projeto seja mantida.
base_dir = Path.cwd().parent

db_path = next(base_dir.rglob("supply_chain_analytics.duckdb"))

conn = duckdb.connect(str(db_path))

# ============================================================
# CRIAÇÃO DA TABELA SILVER PRODUCT MASTER
# ============================================================
#
# Objetivo:
# Criar a tabela silver__product_master__lite a partir da
# camada Bronze, aplicando regras de limpeza e padronização
# nos dados cadastrais dos produtos.
#
# Transformações realizadas:
# - Remoção de espaços excedentes
# - Padronização do SKU para letras maiúsculas
# - Preservação das descrições dos produtos
# - Preservação das categorias e marcas
# - Manutenção do custo unitário do produto
#
# Esta tabela servirá como dimensão de produtos para análises
# de estoque, vendas, movimentações e indicadores financeiros.
#
# ============================================================

conn.execute("""
CREATE OR REPLACE TABLE silver__product_master__lite AS

SELECT

    -- Código único do produto (Stock Keeping Unit)
    -- Padronizado para garantir consistência nos relacionamentos
    -- entre tabelas de estoque, vendas e movimentações
    UPPER(TRIM(sku)) AS sku,

    -- Nome ou descrição do produto
    -- Remove espaços excedentes preservando a grafia original
    TRIM(product_name) AS product_name,

    -- Categoria do produto
    -- Utilizada para segmentações e análises gerenciais
    TRIM(category) AS category,

    -- Marca do produto
    -- Utilizada para análises de desempenho por fabricante
    TRIM(brand) AS brand,

    -- Custo unitário do produto
    -- Base para cálculos financeiros e valuation de estoque
    unit_cost

FROM bronze__product_master__lite;
""")

# Verifica se a tabela foi criada com sucesso
tables = conn.sql("SHOW TABLES").df()

if "silver__product_master__lite" in tables["name"].values:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"✅ Tabela silver__product_master__lite criada com sucesso!"
    )
else:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"❌ Tabela silver__product_master__lite não foi criada!"
    )
conn.close()

[03/06/2026 23:39:10] ✅ Tabela silver__product_master__lite criada com sucesso!


In [5]:
#CRIAÇÃO DA TABELA SILVER SALES TRANSACTIONS
from datetime import datetime
from pathlib import Path
import duckdb

# Procura o banco automaticamente : o notebook continuará encontrando o banco automaticamente dentro do projeto, 
# mesmo que ele seja movido para outra pasta ou máquina, desde que a estrutura do projeto seja mantida.
base_dir = Path.cwd().parent

db_path = next(base_dir.rglob("supply_chain_analytics.duckdb"))

conn = duckdb.connect(str(db_path))

# ============================================================
# CRIAÇÃO DA TABELA SILVER SALES TRANSACTIONS
# ============================================================
#
# Objetivo:
# Criar a tabela silver__sales_transactions__lite a partir da
# camada Bronze, aplicando regras de limpeza, padronização e
# tipagem dos dados transacionais de vendas.
#
# Transformações realizadas:
# - Padronização dos identificadores para maiúsculas
# - Remoção de espaços excedentes
# - Conversão segura do timestamp da transação
# - Criação de uma coluna derivada contendo apenas a data
# - Conversão de quantidade para INTEGER
# - Conversão de preço unitário para DOUBLE
#
# Esta tabela servirá como fato de vendas para análises de
# receita, volume vendido, performance de produtos, canais
# e localidades.
#
# ============================================================

conn.execute("""
CREATE OR REPLACE TABLE silver__sales_transactions__lite AS

SELECT

    -- Identificador único da transação
    -- Padronizado para garantir consistência nos relacionamentos
    UPPER(TRIM(transaction_id)) AS transaction_id,

    -- Data e hora da transação
    -- Tenta converter diretamente para TIMESTAMP.
    -- Caso falhe, tenta interpretar o formato DD/MM/YYYY HH:MM.
    COALESCE(
        TRY_CAST(transaction_ts AS TIMESTAMP),

        TRY_STRPTIME(transaction_ts, '%d/%m/%Y'),

        TRY_STRPTIME(transaction_ts, '%d/%m/%Y %H:%M'),

        TRY_STRPTIME(transaction_ts, '%Y-%m-%d'),

        TRY_STRPTIME(transaction_ts, '%Y-%m-%d %H:%M')
        
    ) AS transaction_ts,

    -- Data derivada da transação
    -- Utilizada para agregações diárias e análises temporais.
    CAST(
        COALESCE(
            TRY_CAST(transaction_ts AS TIMESTAMP),

            TRY_STRPTIME(transaction_ts, '%d/%m/%Y'),

            TRY_STRPTIME(transaction_ts, '%d/%m/%Y %H:%M'),

            TRY_STRPTIME(transaction_ts, '%Y-%m-%d'),

            TRY_STRPTIME(transaction_ts, '%Y-%m-%d %H:%M')
            
        ) AS DATE
    ) AS transaction_date,

    -- Canal de venda
    channel,

    -- Local onde a venda foi realizada
    UPPER(TRIM(location_id)) AS location_id,

    -- SKU do produto vendido
    UPPER(TRIM(sku)) AS sku,

    -- Quantidade vendida
    CAST(qty AS INTEGER) AS qty,

    -- Preço unitário praticado na venda
    CAST(unit_price AS DOUBLE) AS unit_price

FROM bronze__sales_transactions__lite;
""")

# Verifica se a tabela foi criada com sucesso
tables = conn.sql("SHOW TABLES").df()

if "silver__sales_transactions__lite" in tables["name"].values:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"✅ Tabela silver__sales_transactions__lite criada com sucesso!"
    )
else:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"❌ Tabela silver__sales_transactions__lite não foi criada!"
    )
conn.close()

[03/06/2026 23:39:11] ✅ Tabela silver__sales_transactions__lite criada com sucesso!
